# Lorenz sampling training with multiple random seeds

This notebook mirrors `exmaples/lorenz/train_lorenz_sampling.py` but runs multiple training
experiments using different random seeds.

In [1]:
import os
import sys
import datetime
import random
import numpy as np
import pandas as pd
import tensorflow as tf

sys.path.append("../../src")

from example_lorenz_mine_layered import get_lorenz_data
from sindy_utils import library_size
from training_lorenz import train_network

from tensorflow.python.client import device_lib

device_lib.list_local_devices()
print(device_lib.list_local_devices())


/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/szupernikusz/miniconda3/e

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 6218498264362880255
, name: "/device:XLA_CPU:0"
device_type: "XLA_CPU"
memory_limit: 17179869184
locality {
}
incarnation: 13921477586825767217
physical_device_desc: "device: XLA_CPU device"
, name: "/device:XLA_GPU:0"
device_type: "XLA_GPU"
memory_limit: 17179869184
locality {
}
incarnation: 8044166740556694427
physical_device_desc: "device: XLA_GPU device"
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 6917246157
locality {
  bus_id: 1
  links {
  }
}
incarnation: 16549977576150367776
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0"
]


2026-02-01 01:29:38.312588: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-01 01:29:38.312712: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5c4bfed57860 executing computations on platform CUDA. Devices:
2026-02-01 01:29:38.312727: I tensorflow/compiler/xla/service/service.cc:175]   StreamExecutor device (0): NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0
2026-02-01 01:29:38.313392: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-01 01:29:38.313419: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1640] Found device 0 with properties: 
name: NVIDIA GeForce RTX 5060 Laptop GPU major: 12 minor: 0 memoryClockRate(GHz): 1.56
pciBusID: 0000:02:00.0


## Generate training data
This uses the same data generation logic as the training script.

In [2]:
BASE_SEED = 141              # training init seed base (changes per run)
DATA_SEED = 0                # keeps dataset reproducible
OBS_DIM = 50
LATENT_DIM = 3
POLY_ORDER = 3
noise_strength = 1e-3

n_ics_train = 1024
n_ics_val = 20

mlp_depth = 3#4
def slice_data(data, ic_slice):
    """
    Slice along initial-condition axis.
    Works for:
      - per-trajectory arrays: (n_ics, n_steps, dim)
      - flattened arrays: (n_ics*n_steps, dim)
    """
    sliced = {}
    n_steps = data['z'].shape[1]

    for k, v in data.items():
        if isinstance(v, np.ndarray) and v.ndim >= 2 and v.shape[0] == data['z'].shape[0]:
            sliced[k] = v[ic_slice]
        elif isinstance(v, np.ndarray) and v.ndim == 2 and v.shape[0] == data['z'].shape[0] * n_steps:
            ics = np.arange(data['z'].shape[0])[ic_slice]
            idx = np.concatenate([np.arange(i * n_steps, (i + 1) * n_steps) for i in ics])
            sliced[k] = v[idx]
        else:
            sliced[k] = v
    return sliced

print("Start of data generation")
data_all = get_lorenz_data(
    n_ics_train + n_ics_val,
    input_dim=OBS_DIM,
    noise_strength=noise_strength,
    seed=DATA_SEED,
    t_step=0.01,
    n_layers=mlp_depth,
    nested=True,
    max_layers=4,
    normalization=np.array([1/40, 1/40, 1/40]),
)

# Slice data
training_data = slice_data(data_all, slice(0, n_ics_train))
validation_data = slice_data(data_all, slice(n_ics_train, None))
print("End of data generation")

print("Sindy Coefficient generated")
print(training_data['sindy_coefficients'])


Start of data generation
End of data generation
Sindy Coefficient generated
[[  0.          0.          0.       ]
 [-10.         28.          0.       ]
 [ 10.         -1.          0.       ]
 [  0.          0.         -2.6666667]
 [  0.          0.          0.       ]
 [  0.          0.         40.       ]
 [  0.        -40.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]]


## Configure training parameters
Parameters match the script defaults.

In [3]:
params = {}

params['input_dim'] = OBS_DIM
params['latent_dim'] = LATENT_DIM
params['model_order'] = 1
params['poly_order'] = POLY_ORDER
params['include_sine'] = False
params['library_dim'] = library_size(
    params['latent_dim'],
    params['poly_order'],
    params['include_sine'],
    True,
)

# sequential thresholding parameters
params['sequential_thresholding'] = True
params['coefficient_threshold'] = 0.1
params['threshold_frequency'] = 500
params['threshold_start'] = 0
params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
params['nonactive_counter'] = np.zeros((params['library_dim'], params['latent_dim']))
params['coefficient_initialization'] = 'constant'

# loss function weighting
params['loss_weight_decoder'] = 1.0
params['loss_weight_sindy_z'] = 0.0
params['loss_weight_sindy_x'] = 1e-4

params['activation'] = 'sigmoid'
params['widths'] = [64, 32]

# training parameters
params['epoch_size'] = training_data['x'].shape[0]
params['batch_size'] = 1024

params['data_path'] = os.getcwd() + '/'
params['print_progress'] = True
params['print_frequency'] = 50

# Bayesian parameters
params['learning_rate'] = 1e-3
params['prior'] = 'laplace'
params['loss_weight_sindy_regularization'] = 1e-5
params['pi'] = 0.116
params['c_std'] = 20000000.0
params['epsilon'] = 0.2
params['decay'] = 0.02
params['sigma'] = 1.0

# training time cutoffs
params['max_epochs'] = 5001
params['refinement_epochs'] = 1001

print(tf.__version__)


1.14.0


## Train multiple models with different seeds
Update the `seeds` list to run more or fewer experiments.

In [5]:
results = []

for i in range(4):
    seed = BASE_SEED + i
    print(f'EXPERIMENT seed={seed}')

    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

    params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
    params['save_name'] = (
        'lorenz_seed_'
        + str(seed)
        + f"_{mlp_depth}_"
        + datetime.datetime.now().strftime('%Y_%m_%d_%H_%M_%S_%f')
    )

    tf.reset_default_graph()

    results_dict = train_network(training_data, validation_data, params)

    print('training finished')

    results.append({**results_dict, **params, 'seed': seed})

df = pd.DataFrame(results)
df.to_pickle(
    'experiment_results_'
    + datetime.datetime.now().strftime('%Y%m%d%H%M')
    + '.pkl'
)
df


EXPERIMENT seed=141




The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.


Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


TRAINING
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The Ten

2026-02-01 01:30:11.022947: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-01 01:30:11.022980: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1640] Found device 0 with properties: 
name: NVIDIA GeForce RTX 5060 Laptop GPU major: 12 minor: 0 memoryClockRate(GHz): 1.56
pciBusID: 0000:02:00.0
2026-02-01 01:30:11.023015: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcudart.so.10.1
2026-02-01 01:30:11.023023: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcublas.so.10
2026-02-01 01:30:11.023029: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcufft.so.10
2026-02-01 01:30:11.023035: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dy

================= 0  ==================
[[0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]]
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


2026-02-01 01:30:13.843826: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcublas.so.10


Epoch 0
   training loss 0.0013723594602197409, (0.001248719, 0.0, 0.00011731365, 6.326674e-06)
   validation loss 0.0017500383546575904, (0.0016028911, 0.0, 0.00014082048, 6.326674e-06)
decoder loss ratio: 0.014691, decoder SINDy loss  ratio: 0.000100
================= 1  ==================
================= 2  ==================
================= 3  ==================
================= 4  ==================
================= 5  ==================
================= 6  ==================
================= 7  ==================
================= 8  ==================
================= 9  ==================
================= 10  ==================
================= 11  ==================
================= 12  ==================
================= 13  ==================
================= 14  ==================
================= 15  ==================
================= 16  ==================
================= 17  ==================
================= 18  ==================
================= 

...

2026-02-01 10:09:18.327708: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-01 10:09:18.327777: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1640] Found device 0 with properties: 
name: NVIDIA GeForce RTX 5060 Laptop GPU major: 12 minor: 0 memoryClockRate(GHz): 1.56
pciBusID: 0000:02:00.0
2026-02-01 10:09:18.327872: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcudart.so.10.1
2026-02-01 10:09:18.327889: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcublas.so.10
2026-02-01 10:09:18.327905: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcufft.so.10
2026-02-01 10:09:18.327919: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dy

[[0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]]
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
Epoch 0
   training loss 0.000885849935002625, (0.0007433131, 0.0, 0.00013809574, 4.4410936e-06)
   validation loss 0.001218502759002149, (0.0010543884, 0.0, 0.00015967326, 4.4410936e-06)
decoder loss ratio: 0.009664, decoder SINDy loss  ratio: 0.000113
================= 1  ==================
================= 2  ==================
================= 3  ==================
================= 4  ==================
================= 5  ==================
====

2026-02-01 14:40:21.225482: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-01 14:40:21.225811: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1640] Found device 0 with properties: 
name: NVIDIA GeForce RTX 5060 Laptop GPU major: 12 minor: 0 memoryClockRate(GHz): 1.56
pciBusID: 0000:02:00.0
2026-02-01 14:40:21.227256: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcudart.so.10.1
2026-02-01 14:40:21.227345: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcublas.so.10
2026-02-01 14:40:21.227451: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dynamic library libcufft.so.10
2026-02-01 14:40:21.227522: I tensorflow/stream_executor/platform/default/dso_loader.cc:42] Successfully opened dy

================= 0  ==================
[[0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]
 [0.5 0.5 0.5]]
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
Epoch 0
   training loss 0.00116259278729558, (0.0010308931, 0.0, 0.00012496545, 6.7343053e-06)
   validation loss 0.0015342638362199068, (0.0013804798, 0.0, 0.00014704977, 6.7343053e-06)
decoder loss ratio: 0.012652, decoder SINDy loss  ratio: 0.000104
================= 1  ==================
================= 2  ==================
================= 3  ==================
================= 4  ==================
====

KeyboardInterrupt: 